# Page and Section Index Retrieval [Step 1 - Metadata as Index]

> **MLCourse - Agentic AI - Vectorless RAG**

This notebook demonstrates how to build a retrieval system that uses
page numbers and section headings as the primary index. No embeddings
or vectors are involved. We extract text from the Transformer paper
PDF, attach page and section metadata to every chunk, and retrieve
passages by navigating the document structure directly.

### Import all libraries needed for this notebook.


In [ ]:
import re                              # Regex for section detection
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path               # File path handling
from pypdf import PdfReader            # PDF text extraction


### Part 1: Configuration


In [ ]:
# Define paths and constants for the entire notebook.

PDF_PATH = r"D:\projects\python\MLCourse\03_agentic_ai\data\attention_is_all_you_need.pdf"
CHUNK_SIZE = 600                       # Target chunk size in characters
OVERLAP = 100                          # Overlap between adjacent chunks

print(f"PDF path: {PDF_PATH}")
print(f"Chunk size: {CHUNK_SIZE}, overlap: {OVERLAP}")


### Part 2: Extracting Text and Page Metadata


In [ ]:
# We read every page from the PDF and store the raw text along with
# the page number. Page numbers are our first metadata field.

reader = PdfReader(PDF_PATH)
total_pages = len(reader.pages)
print(f"PDF has {total_pages} pages")

# Extract text from each page, stripping Unicode artifacts.
pages_raw = []
for i, page in enumerate(reader.pages):
    text = page.extract_text()
    # Normalize common Unicode issues in academic PDFs.
    text = text.replace("\u2217", "*")     # asterisk-like char
    text = text.replace("\u2019", "'")     # right single quote
    text = text.replace("\u2013", "-")     # en dash
    text = text.replace("\u0142", "l")     # Polish l with stroke
    text = text.replace("\u0105", "a")     # Polish a with ogonek
    pages_raw.append({"page": i, "text": text})
    preview = text[:80].replace("\n", " ")
    print(f"  Page {i}: {len(text)} chars -- {preview}...")


### Part 3: Detecting Section Headings


In [ ]:
# Academic papers follow a predictable section structure. We scan for
# numbered headings and title-like lines to build a section map.

# Common section headings in the Transformer paper.
SECTION_PATTERNS = [
    r"^(?:Abstract)\b",
    r"^\d+\s+Introduction\b",
    r"^\d+\s+Background\b",
    r"^\d+\s+Model Architecture\b",
    r"^\d+\s+Why Self-Attention\b",
    r"^\d+\s+Training\b",
    r"^\d+\s+Results\b",
    r"^\d+\s+Conclusion\b",
    r"^Attention Is All You Need\b",      # Title line
]

section_map = []  # List of (page_index, section_name) tuples.

for page_info in pages_raw:
    page_idx = page_info["page"]
    lines = page_info["text"].split("\n")
    for line in lines:
        line_stripped = line.strip()
        for pattern in SECTION_PATTERNS:
            if re.match(pattern, line_stripped, re.IGNORECASE):
                section_map.append((page_idx, line_stripped))

print("Detected sections:")
for pg, sec in section_map:
    print(f"  Page {pg}: {sec}")


### Part 4: Assigning Sections to Pages


In [ ]:
# Each page belongs to the most recently detected section. We build a
# lookup that maps every page index to its section name.

def build_page_to_section(section_map, total_pages):
    """Map each page to the section it belongs to.

    Walks through detected sections and assigns all pages between
    two consecutive headings to the earlier heading.
    """
    page_section = {}
    for pg, sec in section_map:
        page_section[pg] = sec

    # Forward-fill: each page inherits the section of the nearest
    # previous page that has a detected heading.
    current_section = "Title"
    result = {}
    for pg in range(total_pages):
        if pg in page_section:
            current_section = page_section[pg]
        result[pg] = current_section
    return result

page_to_section = build_page_to_section(section_map, total_pages)

print("Page-to-section mapping:")
for pg in range(total_pages):
    print(f"  Page {pg} -> {page_to_section[pg]}")


### Part 5: Chunking with Metadata


In [ ]:
# We split each page into overlapping chunks and attach metadata
# (page number, section name) to every chunk. This metadata is the
# backbone of our vectorless retrieval system.

chunks = []
for page_info in pages_raw:
    page_idx = page_info["page"]
    text = page_info["text"]
    section = page_to_section[page_idx]

    # Split the page text into overlapping windows.
    start = 0
    while start < len(text):
        end = min(start + CHUNK_SIZE, len(text))
        chunk_text = text[start:end]
        chunks.append({
            "text": chunk_text,
            "page": page_idx,
            "section": section,
            "chunk_id": len(chunks),
            "char_start": start,
            "char_end": end,
        })
        start += CHUNK_SIZE - OVERLAP

print(f"Created {len(chunks)} chunks across {total_pages} pages")
print(f"Avg chunk length: {sum(len(c['text']) for c in chunks) // len(chunks)} chars")


### Part 6: Retrieval by Page Number


In [ ]:
# The simplest form of vectorless retrieval: find all chunks on a
# specific page or range of pages.

def retrieve_by_page(target_page, chunk_list):
    """Return all chunks from the given page number."""
    return [c for c in chunk_list if c["page"] == target_page]

def retrieve_by_page_range(start_page, end_page, chunk_list):
    """Return all chunks within a page range (inclusive)."""
    return [c for c in chunk_list if start_page <= c["page"] <= end_page]

# Test: retrieve all content from page 2.
results = retrieve_by_page(2, chunks)
print(f"Retrieved {len(results)} chunks from page 2")
for c in results:
    preview = c["text"][:60].replace("\n", " ")
    print(f"  [{c['section']}] {preview}...")


### Part 7: Retrieval by Section Name


In [ ]:
# Retrieve all chunks belonging to a named section. This is useful
# when you know which part of the paper contains the answer.

def retrieve_by_section(section_name, chunk_list):
    """Return all chunks whose section matches the given name (case-insensitive)."""
    target = section_name.lower()
    return [c for c in chunk_list if target in c["section"].lower()]

# Test: retrieve all content from the Training section.
results = retrieve_by_section("Training", chunks)
print(f"Retrieved {len(results)} chunks from 'Training' section")
for c in results[:3]:
    preview = c["text"][:60].replace("\n", " ")
    print(f"  [Page {c['page']}] {preview}...")
if len(results) > 3:
    print(f"  ... and {len(results) - 3} more chunks")


### Part 8: Keyword Search Within a Section


In [ ]:
# Combine structural navigation with keyword filtering for precise
# retrieval. We search for a keyword only within a specific section.

def retrieve_by_keyword_in_section(keyword, section_name, chunk_list):
    """Keyword search restricted to a given section."""
    keyword_lower = keyword.lower()
    section_chunks = retrieve_by_section(section_name, chunk_list)
    matches = []
    for c in section_chunks:
        if keyword_lower in c["text"].lower():
            matches.append(c)
    return matches

# Test: find mentions of "attention" in the Model Architecture section.
results = retrieve_by_keyword_in_section("attention", "Model Architecture", chunks)
print(f"Found {len(results)} chunks mentioning 'attention' in Model Architecture")
for c in results[:3]:
    # Show a snippet around the keyword.
    idx = c["text"].lower().find("attention")
    start = max(0, idx - 30)
    end = min(len(c["text"]), idx + 50)
    snippet = c["text"][start:end].replace("\n", " ")
    print(f"  [Page {c['page']}] ...{snippet}...")


### Part 9: Building a Document Navigator


In [ ]:
# We create a helper class that wraps all retrieval functions into a
# single interface. This is the "vectorless database" for our paper.

class PageNavigator:
    """Navigate a document by page numbers and section names.

    No vectors, no embeddings. Pure structural retrieval.
    """

    def __init__(self, chunk_list):
        self.chunks = chunk_list
        self.sections = sorted(set(c["section"] for c in chunk_list))
        self.pages = sorted(set(c["page"] for c in chunk_list))

    def by_page(self, page_num):
        return retrieve_by_page(page_num, self.chunks)

    def by_section(self, section_name):
        return retrieve_by_section(section_name, self.chunks)

    def by_keyword(self, keyword):
        return [c for c in self.chunks if keyword.lower() in c["text"].lower()]

    def by_keyword_in_section(self, keyword, section_name):
        return retrieve_by_keyword_in_section(keyword, section_name, self.chunks)

    def by_page_range(self, start, end):
        return retrieve_by_page_range(start, end, self.chunks)

    def stats(self):
        return {
            "total_chunks": len(self.chunks),
            "total_pages": len(self.pages),
            "sections": self.sections,
        }

nav = PageNavigator(chunks)
info = nav.stats()
print(f"Navigator ready:")
print(f"  Chunks: {info['total_chunks']}")
print(f"  Pages: {info['total_pages']}")
print(f"  Sections: {info['sections']}")


### Part 10: Navigation Demo -- Answering Questions by Page


In [ ]:
# We simulate queries and show how page-based navigation finds the
# right passages without any vector search.

demo_queries = [
    {"question": "What is the Transformer architecture?",
     "hint_page": 2,
     "hint_section": "Model Architecture"},
    {"question": "How is the model trained?",
     "hint_page": 6,
     "hint_section": "Training"},
    {"question": "What results did they achieve?",
     "hint_page": 7,
     "hint_section": "Results"},
]

for q in demo_queries:
    print(f"{'=' * 60}")
    print(f"Q: {q['question']}")
    print(f"  Strategy: navigate to page {q['hint_page']} ({q['hint_section']})")

    # Retrieve by section.
    results = nav.by_section(q["hint_section"])
    # Combine text for context.
    context = " ".join(c["text"][:200] for c in results[:2])
    preview = context[:200].replace("\n", " ")
    print(f"  Found {len(results)} chunks")
    print(f"  Preview: {preview}...")
    print()


### Part 11: Highlighting the Keyword in Context


In [ ]:
# For a given keyword, show exactly where it appears across the
# document with surrounding context.

def highlight_keyword(keyword, chunk_list, context_len=40):
    """Show every occurrence of a keyword with surrounding context."""
    keyword_lower = keyword.lower()
    occurrences = []
    for c in chunk_list:
        text_lower = c["text"].lower()
        idx = 0
        while True:
            idx = text_lower.find(keyword_lower, idx)
            if idx == -1:
                break
            start = max(0, idx - context_len)
            end = min(len(c["text"]), idx + len(keyword) + context_len)
            snippet = c["text"][start:end].replace("\n", " ")
            occurrences.append({
                "page": c["page"],
                "section": c["section"],
                "snippet": snippet,
            })
            idx += 1
    return occurrences

# Find all mentions of "self-attention" across the paper.
occurrences = highlight_keyword("self-attention", chunks)
print(f"Found {len(occurrences)} occurrences of 'self-attention':")
for i, occ in enumerate(occurrences[:6], 1):
    print(f"  {i}. [Page {occ['page']}, {occ['section']}]")
    print(f"     ...{occ['snippet']}...")
print(f"  ({len(occurrences) - 6} more)" if len(occurrences) > 6 else "")


### Part 12: Summary


In [ ]:
# Page and section indexing gives us fast, deterministic retrieval
# with zero embeddings. The key advantages are:
#
# 1. Zero latency: no neural network calls needed.
# 2. Deterministic: same query always returns the same results.
# 3. Interpretable: you know exactly why a passage was retrieved.
# 4. Metadata-rich: page, section, and position are all available.
#
# Limitations:
# - Requires prior knowledge of document structure.
# - Cannot handle semantic queries like "What is similar to X?".
# - Works best with well-structured documents (papers, books, manuals).

print("Vectorless Page/Section Retrieval Summary:")
print("  Extract text and page numbers from PDF")
print("  Detect section headings with regex patterns")
print("  Assign sections to pages via forward-fill")
print("  Chunk text and attach page + section metadata")
print("  Retrieve by page number, section name, or keyword")
print("  No embeddings, no vectors, no latency")
print()
print("Next: Structured Metadata Filtering")
